# Subquestion 2: Car driving among older people

**Question:** How did car driving among people aged 70+ change around the increase in the medical examination age in 2014?

On 1 January 2014, the medical examination age for older driving licence holders increased from 70 to 75. This changed the medical examination requirement, rather than removing the need to renew a licence. [Source: policy decision](https://zoek.officielebekendmakingen.nl/stb-2013-275.html).

This analysis uses CBS dataset 84755NED. It measures **car-driver trips per person per day, for all travel purposes combined**. The main comparison ends in 2019 to avoid the COVID-19 period.

## 1. Data cleaning and validation

The CSV lists the measure above three header rows describing age, transport mode and travel purpose. We use the new dataset 1 with age first and reorder the columns for the analysis. We read these headers, keep the year rows, remove the provisional-data marker (`*`) from year labels and convert decimal commas to numbers. Missing entries remain missing, rather than becoming zero.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Find the data when running from the project folder or Code folder.
folder = Path('Data_age_research')
if not folder.exists():
    folder = Path('../Data_age_research')

file = folder / '1_Mobility_trends_by_trip_(84755NED)_complete.csv'
data = pd.read_csv(file, sep=';', skiprows=5, header=[0, 1, 2],
                   index_col=0, dtype=str)

# Keep years such as 2013*; remove the unit and source rows.
data = data.loc[data.index.str.fullmatch(r'\d{4}\*?')]
data.index = data.index.str.replace('*', '', regex=False).astype(int)
data.index.name = 'Year'
data = data.apply(lambda column: pd.to_numeric(
    column.str.replace(',', '.', regex=False), errors='coerce'))

# The selected measure is fixed in the file header: trips per person per day.
# Reorder the new age-first layout for the selections used below.
data.columns.names = ['Age', 'Mode', 'Purpose']
data = data.reorder_levels(['Mode', 'Age', 'Purpose'], axis=1)
data = data.sort_index()
print('Loaded English dataset:', file.name)

In [ ]:
# These labels match the English CSV categories.
ages = ['50 to 60 years', '60 to 65 years', '65 to 70 years', '70 years or older']
drivers = data.xs(('Car (driver)', 'Total'), axis=1,
                  level=['Mode', 'Purpose'])[ages].copy()
drivers.columns = ['50-59', '60-64', '65-69', '70+']
drivers.columns.name = 'Age group'

validation = pd.DataFrame({
    'Available years': drivers.count(),
    'Missing years': drivers.isna().sum(),
    'Missing (%)': drivers.isna().mean() * 100
})
display(validation)

assert not data.index.duplicated().any(), 'Duplicate years found.'
assert not data.columns.duplicated().any(), 'Duplicate categories found.'
assert not (data < 0).any().any(), 'Negative trip values found.'
assert drivers.index.tolist() == list(range(1999, 2026)), 'Check year coverage.'
assert drivers.notna().all().all(), 'Missing values in the selected driver series.'
print('Validation passed: all four selected series have 27 complete years.')

All selected driver series are complete. Some detailed travel purposes have missing values, but we use the published **Total** purpose for the main analysis. We do not add this total to the individual purposes.

## 2. Driving trends over time

The left graph shows the number of driver trips per person per day. For example, 0.60 means an average of 0.60 trips across everyone in that age group, including people who did not drive that day.

The right graph compares relative changes. We divide each age group's values by its own 2013 value and multiply by 100. Therefore, **100 is the 2013 level**, and 110 means 10% more trips than in 2013. This makes growth easier to compare even when groups start at different levels.

The dashed line marks 2014. Dots show annual estimates; the lines connect these dots. The grey band spans 2019-2022 to include the drop from the pre-COVID value in 2019 to the COVID-affected years 2020-2022. The 2019 boundary is a visual reference, not the start of the pandemic. Younger groups provide context, but they are not automatically valid control groups.

In [ ]:
plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'font.size': 10})
colors = ['#96A5AE', '#4177A0', '#D69B30', '#008577']
indexed = drivers / drivers.loc[2013] * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
drivers.plot(ax=axes[0], color=colors, marker='o', markersize=3)
indexed.plot(ax=axes[1], color=colors, marker='o', markersize=3)
axes[0].set_title('Car-driver trips by age group')
axes[0].set_ylabel('Trips per person per day')
axes[0].set_ylim(bottom=0)
axes[1].set_title('Change relative to 2013')
axes[1].set_ylabel('Index (2013 = 100)')
axes[1].axhline(100, color='grey', linewidth=0.8)

for ax in axes:
    ax.axvline(2014, color='black', linestyle='--', linewidth=1)
    # Include the 2019 baseline so the drop to 2020 falls inside the band.
    ax.axvspan(2019, 2022, color='grey', alpha=0.15)
    ax.set_xlabel('Year')
    ax.grid(alpha=0.2)
    ax.legend(title='Age group')
plt.tight_layout()
plt.show()

**Reading the graphs:** people aged 70+ make fewer driver trips than the younger groups, but their trip frequency increases over time. The index graph shows that this increase already started before 2014. An increase after 2014 alone is therefore not enough to show a policy effect.

## 3. Comparing before and after 2014

We compare the unweighted averages of six annual values before the change (2008-2013) with six after it (2014-2019). We also show 2013 and 2019 separately. These are descriptive comparisons: an existing upward trend can also produce a higher average after 2014.

In [ ]:
before = drivers.loc[2008:2013].mean()
after = drivers.loc[2014:2019].mean()
comparison = pd.DataFrame({
    'Mean 2008-2013': before,
    'Mean 2014-2019': after,
    'Change (%)': (after / before - 1) * 100,
    '2013': drivers.loc[2013],
    '2019': drivers.loc[2019]
})
display(comparison.round(3))

older = drivers['70+']
change = (older.loc[2019] / older.loc[2013] - 1) * 100
print(f'70+: {older.loc[2013]:.2f} trips in 2013 and {older.loc[2019]:.2f} in 2019.')
print(f'This is an increase of {change:.1f}%.')

## 4. Did the increase become faster after 2014?

We fit a straight line to the 2004-2013 values and extend it to 2019. `np.polyfit` calculates the line's slope and intercept; `np.polyval` calculates its predicted values. We also calculate the slope for 2014-2019 to compare the growth rates.

The extended line is a **reference trend**, not a proven prediction of what would have happened without the policy. CBS figures are rounded, modelled trend estimates; this is not a statistical significance test.

In [ ]:
pre_policy = older.loc[2004:2013]
post_policy = older.loc[2014:2019]
pre_line = np.polyfit(pre_policy.index, pre_policy, 1)
post_line = np.polyfit(post_policy.index, post_policy, 1)
years = np.arange(2004, 2020)
reference = np.polyval(pre_line, years)

plt.figure(figsize=(9, 4))
plt.plot(years, older.loc[2004:2019], 'o-', color='#008577', label='Published values, 70+')
plt.plot(years, reference, '--', color='#20364D', label='Extended 2004-2013 trend')
plt.axvline(2014, color='black', linestyle=':', label='Policy change')
plt.xlabel('Year')
plt.ylabel('Driver trips per person per day')
plt.title('Driving among people aged 70+ and the earlier trend')
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f'Annual increase before 2014: {pre_line[0]:.4f} trips per person per day.')
print(f'Annual increase after 2014:  {post_line[0]:.4f} trips per person per day.')
difference = older.loc[2019] - np.polyval(pre_line, 2019)
print(f'2019 difference from the reference trend: {difference:+.4f} trips per person per day.')

The annual increase is about 0.0108 before 2014 and 0.0094 afterwards. This comparison does not show a clear acceleration. The 2019 value is close to the extended earlier trend. Small differences should be interpreted cautiously because the original values are rounded to two decimal places.

## 5. Driving as a share of all trips

To check whether driving increased relative to overall mobility, we divide driver trips by all trips for people aged 70+. This gives the **share of trips made as a driver**, not the percentage of people who drive.

In [ ]:
all_trips = data[('Total', '70 years or older', 'Total')]
assert all_trips.notna().all() and (all_trips > 0).all()
driver_share = older / all_trips * 100

plt.figure(figsize=(9, 4))
plt.plot(driver_share.index, driver_share, color='#008577')
plt.axvline(2014, color='black', linestyle='--')
plt.xlabel('Year')
plt.ylabel('Driver trips / all trips (%)')
plt.title('Share of trips made as a driver: age 70+')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

print(f'Driver share: {driver_share.loc[2013]:.1f}% in 2013 and {driver_share.loc[2019]:.1f}% in 2019.')

The driver share rises from 29.3% to 32.7%. This suggests a shift towards driving, rather than simply more trips overall. However, this shift also started before 2014.

## 6. Travel purposes

The main analysis includes all purposes together. The table below checks the individual purposes for 70+. `NaN` means missing, not zero, so these values cannot provide a complete breakdown of the increase.

In [ ]:
purposes = data.xs(('Car (driver)', '70 years or older'), axis=1,
                   level=['Mode', 'Age']).copy()
purposes = purposes.rename(columns={
    'Total': 'All purposes',
    'To and from work': 'Work',
    'Shopping, groceries': 'Shopping',
    'Education, course, childcare': 'Education and childcare',
    'Leisure': 'Leisure',
    'Other travel purposes': 'Other purposes'
})
purpose_comparison = purposes.loc[[2013, 2019]].T
purpose_comparison['Change'] = purpose_comparison[2019] - purpose_comparison[2013]
display(purpose_comparison.round(2))

Leisure and shopping both show higher driver-trip frequencies in 2019 than in 2013. Missing work and education values mean we cannot explain the full increase by purpose.

## 7. Conclusion

Car driving among people aged 70+ increased from **0.58 to 0.64 trips per person per day between 2013 and 2019 (+10.3%)**. Driving also became a larger share of their trips. However, the increase had already started before 2014, and the trend comparison shows no clear acceleration afterwards.

**The data show an increase in driving, but do not establish that the 2014 policy caused it.** This also does not prove that the policy had no effect. The data combine everyone aged 70+, so the directly relevant 70-74 group cannot be studied separately. Younger groups have different earlier trends, and no uncertainty margins are included. Trip frequency also does not tell us how many individual people drive.

Sources: [CBS dataset 84755NED and definitions](https://www.cbs.nl/nl-nl/cijfers/detail/84755NED), [CBS trend modelling](https://www.cbs.nl/nl-nl/achtergrond/2021/44/tijdreeksmodellering-mobiliteitstrends-1999-2020), [2014 policy decision](https://zoek.officielebekendmakingen.nl/stb-2013-275.html).